# Python Advanced (Rarely Used But Powerful)

---

This notebook covers advanced Python features that are **not commonly used day-to-day** but appear in framework internals, performance-critical code, and metaprogramming. Understanding these helps you read PyTorch, NumPy, and Hugging Face source code.

## Table of Contents
1. [Metaclasses](#1)
2. [Descriptors](#2)
3. [Abstract Base Classes (ABC) Deep Dive](#3)
4. [Protocols (Structural Subtyping)](#4)
5. [Generators & Coroutines Deep Dive](#5)
6. [Memory Management & Garbage Collection](#6)
7. [Slots & Memory Optimization](#7)
8. [C Extensions & Cython Interface](#8)
9. [Operator Overloading Full Guide](#9)
10. [Python Internals Bytecode](#10)
11. [Functional Programming Tools](#11)
12. [weakref Avoid Memory Leaks](#12)
13. [Additional Learning Resources](#13)

---
## 1. Metaclasses <a id='1'></a>

A **metaclass** is the class of a class it controls how classes themselves are created. `type` is the default metaclass for all Python classes.

```
object  ← base of all instances
type    ← metaclass: creates class objects

MyClass = type('MyClass', (object,), {'x': 42})
```

Metaclasses are used in:
- Django ORM (`Model` auto-registers fields)
- `abc.ABCMeta` (enforces abstract methods)
- `enum.EnumMeta`
- PyTorch's `Module` registration

In [1]:
# ===== type() the built-in metaclass =====
# Creating a class dynamically
Dog = type('Dog', (object,), {
    'species': 'Canis lupus familiaris',
    'bark': lambda self: 'Woof!'
})
d = Dog()
print(d.bark())
print(type(Dog))   # <class 'type'>
print(type(d))     # <class '__main__.Dog'>

# ===== Custom metaclass =====
class SingletonMeta(type):
    """Metaclass that ensures only one instance per class."""
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class ModelCache(metaclass=SingletonMeta):
    def __init__(self):
        self._cache = {}

    def get(self, key):
        return self._cache.get(key)

    def set(self, key, value):
        self._cache[key] = value

cache1 = ModelCache()
cache2 = ModelCache()
print(cache1 is cache2)   # True same instance
cache1.set('bert', 'model_weights')
print(cache2.get('bert')) # 'model_weights'

# ===== Metaclass for auto-registration =====
class PluginMeta(type):
    registry = {}

    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        if bases:  # don't register the base class itself
            mcs.registry[name.lower()] = cls
        return cls

class BaseProcessor(metaclass=PluginMeta):
    def process(self, data): pass

class TextProcessor(BaseProcessor):
    def process(self, data): return data.strip().lower()

class ImageProcessor(BaseProcessor):
    def process(self, data): return f'processed_image: {data}'

print(PluginMeta.registry)  # auto-registered by metaclass
proc = PluginMeta.registry['textprocessor']()
print(proc.process('  Hello World  '))

Woof!
<class 'type'>
<class '__main__.Dog'>
True
model_weights
{'textprocessor': <class '__main__.TextProcessor'>, 'imageprocessor': <class '__main__.ImageProcessor'>}
hello world


---
## 2. Descriptors <a id='2'></a>

Descriptors are objects that define `__get__`, `__set__`, `__delete__`. They power `@property`, `@classmethod`, `@staticmethod`, and frameworks like Django/SQLAlchemy.

When you access `obj.attr`, Python first checks if `attr` in `type(obj)` is a descriptor.

In [2]:
# ===== Data descriptor (has __set__) =====
class Validator:
    """Descriptor that validates values on assignment."""
    def __init__(self, min_val=None, max_val=None, dtype=None):
        self.min_val = min_val
        self.max_val = max_val
        self.dtype = dtype
        self.name = None

    def __set_name__(self, owner, name):
        """Called when descriptor is assigned to class attribute."""
        self.name = name
        self.private_name = f'_{name}'

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self  # accessed from class, not instance
        return getattr(obj, self.private_name, None)

    def __set__(self, obj, value):
        if self.dtype and not isinstance(value, self.dtype):
            raise TypeError(f'{self.name} must be {self.dtype.__name__}')
        if self.min_val is not None and value < self.min_val:
            raise ValueError(f'{self.name} must be >= {self.min_val}')
        if self.max_val is not None and value > self.max_val:
            raise ValueError(f'{self.name} must be <= {self.max_val}')
        setattr(obj, self.private_name, value)

class NeuralNet:
    learning_rate = Validator(min_val=1e-7, max_val=1.0, dtype=float)
    dropout       = Validator(min_val=0.0,  max_val=0.9, dtype=float)
    num_layers    = Validator(min_val=1,    max_val=100, dtype=int)

    def __init__(self, lr, dropout, layers):
        self.learning_rate = lr
        self.dropout       = dropout
        self.num_layers    = layers

net = NeuralNet(lr=1e-3, dropout=0.1, layers=6)
print(net.learning_rate, net.dropout, net.num_layers)

try:
    net.learning_rate = 5.0  # > 1.0
except ValueError as e:
    print(e)

try:
    net.dropout = '0.1'  # wrong type
except TypeError as e:
    print(e)

# ===== Non-data descriptor (no __set__) like @staticmethod =====
class CachedProperty:
    """Computes value once and caches it on the instance."""
    def __init__(self, func):
        self.func = func
        self.attrname = None
        self.__doc__ = func.__doc__

    def __set_name__(self, owner, name):
        self.attrname = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        # Store result directly on instance (bypasses descriptor next time)
        val = self.func(obj)
        obj.__dict__[self.attrname] = val
        return val

class Dataset:
    def __init__(self, path):
        self.path = path

    @CachedProperty
    def statistics(self):
        import time
        time.sleep(0.01)  # expensive computation
        return {'mean': 0.5, 'std': 0.2}

ds = Dataset('data.csv')
print(ds.statistics)  # computed
print(ds.statistics)  # cached no recomputation

0.001 0.1 6
learning_rate must be <= 1.0
dropout must be float
{'mean': 0.5, 'std': 0.2}
{'mean': 0.5, 'std': 0.2}


---
## 3. Protocols (Structural Subtyping) <a id='3'></a>

Protocols (PEP 544, Python 3.8+) enable **duck typing with static checking**. Instead of inheriting from an ABC, any class that implements the required methods satisfies the protocol like interfaces in Go or TypeScript.

In [3]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Predictor(Protocol):
    def predict(self, X: list) -> list: ...
    def fit(self, X: list, y: list) -> None: ...

class LinearModel:
    # No inheritance from Predictor just implements the interface
    def fit(self, X, y): self.trained = True
    def predict(self, X): return [0] * len(X)

class RandomForest:
    def fit(self, X, y): self.n_trees = 100
    def predict(self, X): return [1] * len(X)

def evaluate(model: Predictor, X, y):
    model.fit(X, y)
    preds = model.predict(X)
    return sum(p == t for p, t in zip(preds, y)) / len(y)

lm = LinearModel()
rf = RandomForest()

print(isinstance(lm, Predictor))  # True runtime check
print(isinstance(rf, Predictor))  # True

X = [[1, 2], [3, 4]]
y = [0, 1]
print(evaluate(lm, X, y))
print(evaluate(rf, X, y))

True
True
0.5
0.5


---
## 4. Generators & Coroutines Deep Dive <a id='4'></a>

Generators produce values **lazily** (on demand). Coroutines can both receive AND send values. These underpin `asyncio`.

In [4]:
# ===== Generator protocol =====
def infinite_counter(start=0):
    n = start
    while True:
        yield n
        n += 1

gen = infinite_counter()
print([next(gen) for _ in range(5)])  # [0, 1, 2, 3, 4]

# ===== Generator with send() coroutine-style =====
def running_average():
    total, count = 0, 0
    avg = None
    while True:
        value = yield avg   # yield current avg, receive next value
        if value is None:
            break
        total += value
        count += 1
        avg = total / count

coro = running_average()
next(coro)           # prime the generator
print(coro.send(10)) # 10.0
print(coro.send(20)) # 15.0
print(coro.send(30)) # 20.0

# ===== yield from delegate to sub-generator =====
def chain(*iterables):
    for it in iterables:
        yield from it

print(list(chain([1,2], [3,4], [5,6])))  # [1,2,3,4,5,6]

# ===== Generator pipeline (memory efficient) =====
def read_file_lines(path):
    """Stage 1: Read lines lazily."""
    try:
        with open(path) as f:
            yield from f
    except FileNotFoundError:
        yield from iter([])  # empty

def parse_floats(lines):
    """Stage 2: Parse lines to floats."""
    for line in lines:
        try:
            yield float(line.strip())
        except ValueError:
            continue

def clamp(values, low, high):
    """Stage 3: Clamp values."""
    for v in values:
        yield max(low, min(high, v))

# Pipeline only one value in memory at a time
# pipeline = clamp(parse_floats(read_file_lines('big_file.txt')), 0, 1)
# for val in pipeline:
#     process(val)

# ===== itertools advanced iteration =====
import itertools

# chain flatten nested iterables
flat = list(itertools.chain([1,2],[3,4],[5,6]))

# islice lazy slicing of generators
first_5 = list(itertools.islice(infinite_counter(), 5))

# groupby group consecutive elements
data = [('a',1),('a',2),('b',3),('b',4),('c',5)]
for key, group in itertools.groupby(data, key=lambda x: x[0]):
    print(key, list(group))

# product cartesian product (grid search!)
params = list(itertools.product([0.001, 0.01], [32, 64], ['adam', 'sgd']))
print(f'Grid search: {len(params)} combinations')

# combinations / permutations
print(list(itertools.combinations('ABC', 2)))
print(list(itertools.permutations('AB', 2)))

[0, 1, 2, 3, 4]
10.0
15.0
20.0
[1, 2, 3, 4, 5, 6]
a [('a', 1), ('a', 2)]
b [('b', 3), ('b', 4)]
c [('c', 5)]
Grid search: 8 combinations
[('A', 'B'), ('A', 'C'), ('B', 'C')]
[('A', 'B'), ('B', 'A')]


---
## 5. Memory Management & Garbage Collection <a id='5'></a>

Python uses **reference counting** as the primary memory management mechanism. When an object's reference count drops to 0, it is immediately freed. A **cyclic garbage collector** handles reference cycles.

In [5]:
import gc
import sys
import weakref

# ===== Reference counting =====
a = [1, 2, 3]    # refcount = 1
b = a             # refcount = 2
c = a             # refcount = 3
print(sys.getrefcount(a))  # 4 (getrefcount itself adds 1 temporarily)

del b             # refcount = 3
del c             # refcount = 2
print(sys.getsizeof(a))   # bytes used

# ===== Garbage collector (handles cycles) =====
gc.enable()       # on by default
gc.collect()      # manual collection

# Example of reference cycle (would leak without cyclic GC)
class Node:
    def __init__(self):
        self.next = None

n1, n2 = Node(), Node()
n1.next = n2
n2.next = n1   # cycle!
del n1, n2     # refcount > 0 due to cycle, but cyclic GC collects it
collected = gc.collect()
print(f'Collected {collected} objects')

# ===== weakref reference without incrementing refcount =====
class HeavyModel:
    def __init__(self, name):
        self.name = name
        self.weights = list(range(10000))

model = HeavyModel('ResNet')
ref   = weakref.ref(model)       # weak reference

print(ref())              # HeavyModel still alive
print(ref().name)

del model                 # delete strong reference
gc.collect()
print(ref())              # None object was freed

# ===== __del__ finalizer (use sparingly) =====
class GPUResource:
    def __init__(self):
        print('Allocating GPU memory')

    def __del__(self):
        print('Freeing GPU memory')  # called when reference count = 0

r = GPUResource()
del r  # triggers __del__

4
88
Collected 2 objects
ResNet
None
Allocating GPU memory
Freeing GPU memory


---
## 6. Operator Overloading Full Guide <a id='6'></a>

Python's **dunder (magic) methods** allow you to define how operators work on your objects. NumPy and PyTorch tensors use this extensively.

| Method | Operator | Called by |
|--------|----------|----------|
| `__add__` | `+` | `a + b` |
| `__mul__` | `*` | `a * b` |
| `__matmul__` | `@` | `a @ b` (matrix multiply) |
| `__getitem__` | `[]` | `a[i]` |
| `__len__` | `len()` | `len(a)` |
| `__iter__` | `for x in a` | iteration |
| `__contains__` | `in` | `x in a` |
| `__enter__/__exit__` | `with` | context manager |
| `__call__` | `()` | `obj()` |
| `__lt__,__gt__,...` | comparison | `a < b` |

In [6]:
class Tensor:
    """Mini tensor class demonstrating operator overloading."""
    def __init__(self, data: list):
        self.data = list(data)

    def __repr__(self):
        return f'Tensor({self.data})'

    def __len__(self):          return len(self.data)
    def __getitem__(self, idx): return self.data[idx]
    def __setitem__(self, idx, val): self.data[idx] = val
    def __iter__(self):         return iter(self.data)
    def __contains__(self, val): return val in self.data

    def __add__(self, other):
        if isinstance(other, Tensor):
            return Tensor([a + b for a, b in zip(self.data, other.data)])
        return Tensor([a + other for a in self.data])  # scalar broadcast

    def __radd__(self, other):   return self.__add__(other)  # other + self
    def __sub__(self, other):    return Tensor([a - b for a, b in zip(self.data, other.data)])
    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return Tensor([a * other for a in self.data])
        return Tensor([a * b for a, b in zip(self.data, other.data)])
    def __rmul__(self, other):   return self.__mul__(other)
    def __neg__(self):           return Tensor([-a for a in self.data])  # -tensor
    def __abs__(self):           return Tensor([abs(a) for a in self.data])

    def __matmul__(self, other): # dot product: t1 @ t2
        return sum(a * b for a, b in zip(self.data, other.data))

    def __eq__(self, other):
        if isinstance(other, Tensor):
            return all(a == b for a, b in zip(self.data, other.data))
        return False

    def __call__(self, x):       # makes object callable: tensor(x)
        return [v * x for v in self.data]

    def __bool__(self):
        return any(self.data)     # truthy if any non-zero element


t1 = Tensor([1, 2, 3])
t2 = Tensor([4, 5, 6])

print(t1 + t2)      # Tensor([5, 7, 9])
print(t1 * 2)       # Tensor([2, 4, 6])
print(3 * t1)       # Tensor([3, 6, 9]) uses __rmul__
print(t1 @ t2)      # 32  (dot product)
print(-t1)          # Tensor([-1, -2, -3])
print(t1[1])        # 2
print(2 in t1)      # True
print(len(t1))      # 3
print(t1(10))       # [10, 20, 30] callable
print(bool(t1))     # True

Tensor([5, 7, 9])
Tensor([2, 4, 6])
Tensor([3, 6, 9])
32
Tensor([-1, -2, -3])
2
True
3
[10, 20, 30]
True


---
## 7. Python Internals Bytecode <a id='7'></a>

Python compiles source code to **bytecode** before execution. Understanding this helps explain performance and debug subtle behaviors.

In [7]:
import dis

# ===== Inspect bytecode =====
def add_numbers(a, b):
    return a + b

dis.dis(add_numbers)  # print bytecode instructions

# ===== List comp vs loop bytecode shows why comp is faster =====
def list_comp():
    return [x**2 for x in range(10)]

def loop():
    result = []
    for x in range(10):
        result.append(x**2)
    return result

print('=== List comprehension ===')
dis.dis(list_comp)

# ===== Code object attributes =====
code = add_numbers.__code__
print(f'\nFunction: {add_numbers.__name__}')
print(f'Args:     {code.co_varnames[:code.co_argcount]}')
print(f'Locals:   {code.co_varnames}')
print(f'Stack size: {code.co_stacksize}')
print(f'Bytecode: {code.co_code}')

  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (a)
              4 LOAD_FAST                1 (b)
              6 BINARY_OP                0 (+)
             10 RETURN_VALUE
=== List comprehension ===
 10           0 RESUME                   0

 11           2 LOAD_GLOBAL              1 (NULL + range)
             12 LOAD_CONST               1 (10)
             14 CALL                     1
             22 GET_ITER
             24 LOAD_FAST_AND_CLEAR      0 (x)
             26 SWAP                     2
             28 BUILD_LIST               0
             30 SWAP                     2
        >>   32 FOR_ITER                 7 (to 50)
             36 STORE_FAST               0 (x)
             38 LOAD_FAST                0 (x)
             40 LOAD_CONST               2 (2)
             42 BINARY_OP                8 (**)
             46 LIST_APPEND              2
             48 JUMP_BACKWARD            9 (to 32)
        >>   50 END_FOR
       

---
## 8. Functional Programming Tools <a id='8'></a>

In [8]:
from functools import reduce, partial, lru_cache, wraps, singledispatch
import operator

# ===== partial fix some arguments =====
def log_softmax(x, temperature=1.0):
    import math
    scaled = [xi / temperature for xi in x]
    max_x  = max(scaled)
    exp_x  = [math.exp(xi - max_x) for xi in scaled]
    total  = sum(exp_x)
    return [math.log(e / total) for e in exp_x]

cold_softmax = partial(log_softmax, temperature=0.5)
hot_softmax  = partial(log_softmax, temperature=2.0)
print(cold_softmax([1.0, 2.0, 3.0]))

# ===== reduce fold a sequence =====
factorial = reduce(operator.mul, range(1, 6))  # 5! = 120
print(factorial)

# ===== singledispatch function overloading =====
@singledispatch
def process(data):
    raise TypeError(f'Unsupported type: {type(data)}')

@process.register(str)
def _(data: str):
    return data.strip().lower().split()

@process.register(list)
def _(data: list):
    return [x for x in data if x is not None]

@process.register(dict)
def _(data: dict):
    return {k: v for k, v in data.items() if v is not None}

print(process('  Hello World  '))
print(process([1, None, 2, None, 3]))
print(process({'a': 1, 'b': None, 'c': 3}))

# ===== compose functions =====
def compose(*funcs):
    """f = compose(f1, f2, f3) → f(x) = f1(f2(f3(x)))"""
    def composed(x):
        for f in reversed(funcs):
            x = f(x)
        return x
    return composed

pipeline = compose(
    lambda x: [v / max(x) for v in x],  # normalize
    lambda x: [v for v in x if v > 0],   # filter
    lambda x: [v * 2 for v in x]         # scale
)
print(pipeline([-1, 0, 2, 4, 8]))

[-4.142931628499899, -2.142931628499899, -0.1429316284998995]
120
['hello', 'world']
[1, 2, 3]
{'a': 1, 'c': 3}
[0.25, 0.5, 1.0]


---
## 9. Additional Learning Resources <a id='13'></a>

### Python Internals
- **CPython Internals** (Anthony Shaw): https://realpython.com/products/cpython-internals-book/
- **Python Data Model**: https://docs.python.org/3/reference/datamodel.html
- **Internals Documentation**: https://devguide.python.org/internals/

### Advanced Python
- **Fluent Python** (Luciano Ramalho): https://www.oreilly.com/library/view/fluent-python-2nd/9781492056348/
- **Python 3 Patterns and Idioms** (Bruce Eckel): https://python-3-patterns-idioms-test.readthedocs.io/
- **Python Cookbook** (Beazley & Jones): https://www.oreilly.com/library/view/python-cookbook-3rd/9781449357337/

### Metaclasses
- **Python Metaclasses** (Real Python): https://realpython.com/python-metaclasses/
- **PEP 3115 Metaclasses**: https://peps.python.org/pep-3115/

### Descriptors
- **Descriptor HowTo Guide** (Python docs): https://docs.python.org/3/howto/descriptor.html
- **Python Descriptors** (Real Python): https://realpython.com/python-descriptors/

### Functional Programming
- **Functional Programming HOWTO**: https://docs.python.org/3/howto/functional.html
- **toolz library**: https://toolz.readthedocs.io/en/latest/
- **fn.py**: https://github.com/kachayev/fn.py

### Generators & Coroutines
- **PEP 342 Coroutines via Enhanced Generators**: https://peps.python.org/pep-0342/
- **Generators Deep Dive** (David Beazley): http://www.dabeaz.com/generators/
- **A Curious Course on Coroutines** (David Beazley): http://www.dabeaz.com/coroutines/